## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker.
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

***Important Note:*** The current LangChain setup is reading the CSV and JSON files as plain text and chunking them without explicitly modeling the relational structure between tables like matches.csv and teams.csv. As a result, the LLM treats each chunk as isolated text and doesn’t infer foreign key relationships, which are essential to answer questions like “Which teams played a particular match?”. This setup could be more suitable for non-relational databases. 

### Imports

In [1]:
import os
import glob
from dotenv import load_dotenv
import gradio as gr

# imports from langchain, Chroma

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.embeddings import HuggingFaceEmbeddings

# imports for plotting

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go

### Constants

In [2]:
MODEL = 'gpt-4o-mini'
db_name = 'vector_db'

### Load API key

In [3]:
load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

### Read in dataset using Langchain's loaders

In [4]:
folder = "assets/football_dataset"

def add_metadata(doc, doc_type):
    doc.metadata["doc_type"] = str(doc_type)
    return doc

text_loader_kwargs = {'encoding': 'utf-8'}

documents = []
csv_loader = DirectoryLoader(folder, glob = "*.csv", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)

json_loader = DirectoryLoader(folder, glob="*.json", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)

csv_docs = csv_loader.load() 
json_docs = json_loader.load()

documents = csv_docs + json_docs

for doc in documents:
    add_metadata(doc, os.path.splitext(os.path.basename(doc.metadata["source"]))[0])



In [ ]:
documents

### Create chunks

RecursiveCharacterTextSplitter- Best for semi-structured or structured text (e.g. CSVs, JSON, Markdown)

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Total number of chunks: {len(chunks)}")
print(f"Document types found: {set(doc.metadata['doc_type'] for doc in documents)}")


Total number of chunks: 372
Document types found: {'matches', 'dataset-metadata', 'scores', 'coaches', 'referees', 'leagues', 'players', 'seasons', 'teams', 'stadiums', 'standings'}


### Create embeddings

In [7]:
# Put the chunks of data into a Vector Store that associates a Vector Embedding with each chunk
# Chroma is a popular open source Vector Database 

embeddings = OpenAIEmbeddings()

# delete of already exists
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# create vector store
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 372 documents


In [8]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 372 vectors with 1,536 dimensions in the vector store


### Set up the conversation chain with langchain

In [ ]:
# create new chat with OpenAI
llm = ChatOpenAI(temperature=0.7, model=MODEL)

# for using ollama locally
# llm = ChatOpenAI(temperature=0.7, model='llama3.2', base_url='http://localhost:11434/v1', api_key='ollama')

# set up conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# set up retriever- used for high level abstraction in RAGs
retriever = vectorstore.as_retriever()

# Put it all together
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

In [10]:
# Example Queries

#query = "In which country was the Serie A league held?"
query = "when was Real madrid club founded?"
result = conversation_chain.invoke({"question": query})
print(result["answer"])

Real Madrid Club de Fútbol was founded in 1902.


### Setup Gradio

In [11]:
def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)